[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day30-gptq-quantization.ipynb)

# Day 30 — GPTQ: Second-Order Weight Quantization

Quantize one weight at a time and compensate the rest with the inverse Hessian. We do it on a tiny toy model by hand (full GPTQ math, no library), then run real AutoGPTQ 4-bit group-128 on Qwen2.5-1.5B and measure perplexity before/after. T4 GPU required for part 2; part 1 runs on CPU in seconds.

In [ ]:
# Cell 1: one-time installs. Requires a T4 GPU runtime (Runtime > Change runtime type).
pip install --quiet auto-gptq optimum datasets
# Expected output: successfully installed ... (takes ~2 min)

## Part 1 — The OBQ compensation step by hand (CPU, ~5 min)

Single output neuron, 2 weights. Quantize w0 to a coarse grid, then compensate w1 using H-inverse.

In [ ]:
import numpy as np

# Calibration activations: channel 0 is 30x larger than channel 1
X = np.array([[3.0, 0.1]])          # shape (1, 2)
H = 2 * X.T @ X                     # Hessian ~ 2XX^T
print("H =\n", H)
Hinv = np.linalg.inv(H + 1e-6*np.eye(2))   # dampened inverse
print("Hinv =\n", Hinv)

w = np.array([0.452, -0.731])       # true weights
grid_step = 0.1                     # coarse fake quantization grid

# Quantize w0 (left-to-right order, like GPTQ)
q0 = round(w[0]/grid_step)*grid_step
e = w[0] - q0                       # quantization error
print(f"w0={w[0]:.3f} -> q0={q0:.3f}, error e={e:+.4f}")

# OBQ compensation on remaining weight w1
delta_w1 = -(Hinv[1,0]/Hinv[0,0]) * e
w1_comp = w[1] + delta_w1
print(f"w1 compensated: {w[1]:.4f} -> {w1_comp:.4f} (delta {delta_w1:+.4f})")

# Compare output error: RTN (no compensation) vs GPTQ (compensation)
x = np.array([3.0, 0.1])
y_true = w @ x
err_rtn  = abs(np.array([q0, w[1]]) @ x - y_true)
err_gptq = abs(np.array([q0, w1_comp]) @ x - y_true)
print(f"true output {y_true:.4f}")
print(f"RTN output error : {err_rtn:.6f}")
print(f"GPTQ output error: {err_gptq:.6f}")
# Expected output:
# w0=0.452 -> q0=0.500, error e=-0.0480
# w1 compensated: -0.7310 -> -0.7294 (delta +0.0016)
# RTN output error : ~0.144 ; GPTQ output error: ~0.0005 (orders of magnitude smaller)

## Part 2 — Real GPTQ on Qwen2.5-1.5B (T4, ~20-30 min)

Baseline perplexity -> quantize with 128 WikiText calibration samples -> perplexity again. 7B+ is read-only here: it needs ~20 GB.

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
# Expected output on T4: cuda available: True / Tesla T4

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset
import torch

model_id = "Qwen/Qwen2.5-1.5B"
tok = AutoTokenizer.from_pretrained(model_id)
tok.pad_token = tok.eos_token

# 128 calibration samples of 2048 tokens from WikiText-2 train
cal = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
texts = [t for t in cal["text"] if len(t) > 200][:200]
examples = []
for t in texts:
    ids = tok(t, return_tensors="pt").input_ids[0][:2048]
    if len(ids) == 2048:
        examples.append(torch.stack([ids, torch.ones_like(ids)]))
    if len(examples) == 128:
        break
print("calibration examples:", len(examples))
# Expected output: calibration examples: 128

In [ ]:
from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig
import time

cfg = BaseQuantizeConfig(bits=4, group_size=128, dataset="wikitext2", desc_act=False)
model = AutoGPTQForCausalLM.from_pretrained(model_id, cfg)
t0 = time.time()
model.quantize(examples)          # builds H = 2XX^T per layer, quantizes + compensates
print(f"quantize wall time: {(time.time()-t0)/60:.1f} min")
# Expected output: quantize wall time: ~3-8 min on T4

In [ ]:
import math
from torch.nn import CrossEntropyLoss

def perplexity(m, tokenizer, n_seqs=50, seqlen=512):
    test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    enc = tokenizer("\n\n".join(test["text"]), return_tensors="pt")
    ids = enc.input_ids[:, : n_seqs*seqlen]
    loss_fct = CrossEntropyLoss()
    nlls = []
    m.eval()
    with torch.no_grad():
        for i in range(0, ids.size(1), seqlen):
            chunk = ids[:, i:i+seqlen].to(m.device if hasattr(m, 'device') else 'cuda')
            out = m(chunk, labels=chunk)
            nlls.append(out.loss)
    return math.exp(torch.stack(nlls).mean().item())

ppl_q = perplexity(model.model, tok)
print(f"GPTQ-4bit perplexity: {ppl_q:.2f}")
# Expected output: GPTQ-4bit perplexity: ~8.4-8.9 (within 0.3 of fp16 baseline ~8.2-8.6)

In [ ]:
# fp16 baseline for comparison (reload fresh fp16 model, skip if short on time)
from transformers import AutoModelForCausalLM
fp16 = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
ppl_fp16 = perplexity(fp16, tok)
print(f"fp16 perplexity: {ppl_fp16:.2f}  |  GPTQ-4bit: {ppl_q:.2f}  |  delta: {ppl_q-ppl_fp16:+.2f}")
# Expected output: fp16 perplexity: ~8.3 | GPTQ-4bit: ~8.6 | delta: +0.2-0.3

In [ ]:
# Save, reload, and sanity-generate 50 tokens
model.save_quantized("./qwen2.5-1.5b-gptq-4bit")
reloaded = AutoGPTQForCausalLM.from_quantized("./qwen2.5-1.5b-gptq-4bit", device="cuda:0")
inp = tok("The future of AI inference is", return_tensors="pt").to("cuda:0")
out = reloaded.generate(**inp, max_new_tokens=50, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))
# Expected output: a coherent 50-token continuation, no repeated-token loops

## Wrap-up — record your numbers

Copy these into the "What to measure" table in the Day 30 PDF:

| Metric | Your number |
|---|---|
| fp16 perplexity | |
| GPTQ 4-bit g128 perplexity | |
| delta ppl (target < 0.3) | |
| quantize wall time | |
| peak RAM during quantize | |
| 50-token generation coherent? | |

**Checkpoints:** (1) GPTQ minimizes output error delta-transpose H delta, not per-weight error. (2) Remaining weights get minus (H-inverse column / diagonal) times e. (3) Calibration data builds H = 2 X X-transpose, deciding which weights the data exercises.